In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

### Deux interfaces d'interaction avec les LLMs
### LLMs (input : string prompt) >>>> from langchain_openai.llms import OpenAI
### Chat models : ChatGroq; ChatOpenAI()

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=200)
#Todo : Tester une interaction avec le modèle



In [4]:
###Les LLms sont des Models Sans Etat
#Todo : poser au modèle une question en relation avec le contexte précédent

"Je suis désolé, mais je ne peux pas connaître votre profession spécifique sans plus d'informations. Pourriez-vous me donner quelques détails sur votre travail ou vos responsabilités ? Cela m'aidera à vous donner une réponse plus précise."

In [ ]:
### L'interface "Chat"
"""
System role : Used for instructions the model should use to answer a user question
User role : Used for the user’s query and any other content produced by the user
Assistant role : Used for content generated by the model
"""

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
#Todo : créer un message system définissant le role de l'agent et un message utilisateur lui est destiné
system = ""
human = ""

prompt = [system, human]

resp = model.invoke(prompt)
print(resp.content)


In [ ]:
#stateless models
#Todo : Tester et fixer la mémoire de l'agent
user = HumanMessage("")


In [8]:
#Prompt Template : Making LLM Prompts Reusable
#LangChain provides prompt template interfaces that make it easy to construct prompts with dynamic inputs
from langchain_core.prompts import PromptTemplate
template = PromptTemplate.from_template(
    """
Answer the question based on the context below.
If the question cannot be answered using the information provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer:
    """
)

res = template.invoke(
    {
    "context": """The most recent advancements in NLP are being driven by LargeLanguage Models (LLMs).
    These models outperform their smaller counterparts and have become invaluable for developers
    who are creating applications with NLP capabilities. Developers can tap into these models through
    Hugging Face's `transformers` library, or by utilizing OpenAI and Cohere's offerings through
    the `openai` and `cohere` libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
    }
)


In [ ]:
completion = model.invoke(res)
completion

In [15]:
### If you’re looking to build an AI chat application, the ChatPromptTemplate can be
### used instead to provide dynamic inputs based on the role of the chat message:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages(
    [
        ('system', '''Answer the question based on the context below. If the question
        cannot be answered using the information provided, answer with "I don\'t know".'''),
        ('human', 'Context: {context}'),
        ('human', 'Question: {question}'),
    ]
)

In [ ]:
model.invoke(template.invoke(
    {
        "context": """The most recent advancements in NLP are being driven by Large Language Models (LLMs).
        These models outperform their smaller counterparts and have become invaluable for
        developers who are creating applications with NLP capabilities.
        Developers can tap into these models through Hugging Face's `transformers` library,
        or by utilizing OpenAI and Cohere's offerings through the `openai` and `cohere`libraries, respectively.""",
        "question": "Which model providers offer LLMs?s"
    }
))

In [20]:
#Getting Specific Formats out of LLMs
from pydantic import BaseModel
#Todo : créer yne classe RepondreAvecJustification ==> Schema de sortie

In [21]:
#specifier le schema de sortie (a la place de 'None')
structured_model = model.with_structured_output(None)

In [ ]:
structured_model.invoke("""What weighs more, a pound of bricks or a pound of feathers?""")

In [26]:
### Combinaison des différents blocks de langchain
### L'interface Runnable

In [ ]:
# — invoke: transforms a single input into an output
# — batch: efficiently transforms multiple inputs into multiple outputs
# — stream: streams output from a single input as it’s produced
# - In Python, each of the three methods have asyncio equivalents.
completion = model.batch(["Salut", "Aurevoir"])
completion

In [ ]:
for token in model.stream("c'est quoi is the result of 1 + 1 ?"):
    print(token)

### Chainage

In [39]:
#chainage
from langchain_core.runnables import chain

template = ChatPromptTemplate.from_messages(
    [
        ('system', '''vous etes un assistant utile qui répond brièvement à la question de l'utilisateur'''),
        ('human', '{question}'),
    ]
)

@chain
def chainer(values):
    prompt = template.invoke(values)
    return model.invoke(prompt)

res = chainer.invoke({'question' : "c'est qui mon nom?"})

In [ ]:
#Declarative Composition
#Todo : une version declaratve du chainage


In [ ]:
#Json Output
#Todo :
"""
Créez un pipeline LangChain qui utilise JsonOutputParser pour forcer un modèle à répondre
sous forme de JSON valide en injectant dynamiquement les instructions de formatage via get_format_instructions() dans un ChatPromptTemplate,
puis enchaînant le prompt, le LLM et le parser avec l'opérateur | pour obtenir un
dictionnaire Python exploitable."""